# RQ1 TextVQA: reference audit and input-stage calibration

Enable Internet and **T4 ×2**. Reuse the existing `rq1-inputs-v1.zip` or its attached extracted Kaggle Dataset. No new images, input ZIP or dependency versions are needed. This notebook runs project instrumentation around unchanged native ApET/LLaVA.

For the first run, enable **RUN_TARGET_AUDIT** and **RUN_CALIBRATION** only. The audit checks all six frozen targets; calibration measures the first loss/control pair × 32 patches. Keep the full pilot off until the downloaded results are reviewed. All default flags are False to prevent automatic inference on notebook import. See `research/RQ1_TEXTVQA_INPUT_PILOT.md` for cells to paste into an existing live notebook.


In [ ]:
import json
import shutil
import subprocess
import zipfile
from datetime import datetime, timezone
from pathlib import Path

RUN_TARGET_AUDIT = False  # First run: True.
RUN_CALIBRATION = False   # First run: True; requires the audit to pass.
RUN_FULL_PILOT = False    # Later only, after reviewing the 2-question calibration.
AUDIT_DIR = None          # Optional Path to an existing extracted audit run.
CALIBRATION_DIR = None    # Optional Path to an existing extracted calibration run.
REPO_URL = "https://github.com/doguineyes/CSE590-ApET.git"
REPO_REF = "apet-analysis"
INPUT_ZIP = None          # Optional exact ZIP or extracted bundle path.
CHECKPOINT_ARGS = []      # Optional local original-format LLaVA/CLIP model paths.

if not any([RUN_TARGET_AUDIT, RUN_CALIBRATION, RUN_FULL_PILOT]):
    raise RuntimeError("Set RUN_TARGET_AUDIT=True and RUN_CALIBRATION=True for the first run; leave RUN_FULL_PILOT=False.")
print("Preparing code; inference has not started.", flush=True)
repo = Path("/kaggle/working/CSE590-ApET")
if not repo.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True, timeout=120)
subprocess.run(["git", "fetch", "origin"], cwd=repo, check=True, timeout=120)
subprocess.run(["git", "checkout", REPO_REF], cwd=repo, check=True, timeout=120)
if REPO_REF == "apet-analysis":
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True, timeout=120)
subprocess.run(["git", "rev-parse", "HEAD"], cwd=repo, check=True)
original_python = repo / "environments/original/.venv/bin/python"
inputs = Path("/kaggle/working/rq1-inputs-v1")
print("Repository:", repo, flush=True)


## Reuse inputs and the original environment

This leaves the notebook kernel unchanged. A live session reuses its environment and checkpoint cache; a fresh session installs the same locked environment. Torch-dependent prerequisite tests must run here rather than on the Intel Mac.


In [ ]:
if not (inputs / "protocol.json").exists():
    input_zip = Path(INPUT_ZIP) if INPUT_ZIP else None
    if input_zip is None:
        zip_candidates = list(Path("/kaggle/input").rglob("rq1-inputs-v1.zip"))
        zip_candidates += list(Path("/kaggle/working").glob("rq1-inputs-v1.zip"))
        extracted = []
        for path in Path("/kaggle/input").rglob("protocol.json"):
            try:
                if json.loads(path.read_text()).get("version") == "rq1-input-zero-v1":
                    if all((path.parent / name / "manifest.json").is_file() for name in ["gqa", "mmstar", "textvqa"]):
                        extracted.append(path.parent)
            except (ValueError, OSError):
                pass
        if len(zip_candidates) == 1:
            input_zip = zip_candidates[0]
        elif len(extracted) == 1:
            inputs = extracted[0]  # Kaggle may unpack a ZIP uploaded as a Dataset.
        else:
            raise RuntimeError(f"Attach rq1-inputs-v1.zip or its extracted files, and set INPUT_ZIP if needed. ZIPs: {zip_candidates}; folders: {extracted}")
    if input_zip is not None and input_zip.is_dir():
        inputs = input_zip
    elif input_zip is not None:
        inputs.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(input_zip) as z:
            for item in z.infolist():
                destination = (inputs / item.filename).resolve()
                if inputs.resolve() not in destination.parents:
                    raise ValueError("Unexpected ZIP path")
            z.extractall(inputs)
print("Accepted input directory:", inputs)
print(json.loads((inputs / "protocol.json").read_text()))
subprocess.run([
    "bash", "bootstrap/setup_original_linux.sh", "--require-cuda", "--expected-gpus", "2",
    "--output", "/kaggle/working/rq1-runtime.json",
], cwd=repo, check=True)
# Explicit import prevents Torch-dependent tests silently skipping on Kaggle.
subprocess.run([str(original_python), "-I", "-c", "import torch; print(torch.__version__)"], check=True)
for test in ["research/tests/test_rq1_input.py", "research/tests/test_rq1_textvqa_runner.py", "research/tests/test_rq1_native.py"]:
    subprocess.run([str(original_python), "-I", test], cwd=repo, check=True)


## Logging helper

Each run gets a new timestamped folder, ZIP and external log. A failed run still produces its evidence ZIP; download it and the log before ending the session. Do not pull new code between the audit and interventions: a changed measurement script invalidates the audit and requires a new one.


In [ ]:
def run_experiment(label, manifest, options):
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    folder = Path(f"/kaggle/working/{label}-{stamp}")
    command = [str(original_python), "-I", "scripts/run_rq1_input.py",
               "--manifest", str(inputs / manifest), "--output", str(folder),
               "--download-budget-gib", "20", *CHECKPOINT_ARGS, *options]
    print("Run directory:", folder, flush=True)
    log = folder.with_suffix(".log")
    with log.open("w") as stream:
        process = subprocess.Popen(command, cwd=repo, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                stream.write(line)
                stream.flush()
            status = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=20)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
    if (folder / "report.json").exists():
        report = json.loads((folder / "report.json").read_text())
        print("Status:", report["status"], "wall seconds:", round(report["wall_seconds"], 1))
        for question in report.get("questions", []):
            print(question)
    print("Download:", folder.with_suffix(".zip"), "and", log)
    if status:
        raise RuntimeError(f"Run failed (exit {status}); download the report ZIP/log before restarting.")
    return folder



## 1. Audit all six fixed reference answers

No interventions. Check literal and initial-capital annotation spellings against native generated tokens. The generated-answer variant is diagnostic only. The fixed initial-capital policy must get full consensus credit, exact prefix alignment and score repeatability within 0.001 nats per token. No automatic retargeting or replacement of failed cases.


In [ ]:
if RUN_TARGET_AUDIT:
    AUDIT_DIR = run_experiment(
        "rq1-textvqa-target-audit-v1", "textvqa/manifest.json",
        ["--mode", "target-audit", "--questions", "6"],
    )


## 2. Calibration: Brolio loss + Dominion preserved control

Two questions × 32 singleton patch interventions. The runner verifies the six-case audit before loading weights, then requires fresh baseline full credit and exact reference alignment. Repeats, no-op and native/cached parity precede scoring; the final reference checks drift. Review ZIP/log before expanding.


In [ ]:
if RUN_CALIBRATION:
    if AUDIT_DIR is None:
        raise RuntimeError("Run the six-case audit first, or set AUDIT_DIR to its extracted run folder.")
    CALIBRATION_DIR = run_experiment(
        "rq1-textvqa-calibration-v1", "textvqa/manifest.json",
        ["--questions", "2", "--tokens", "32",
         "--target-audit-report", str(Path(AUDIT_DIR) / "report.json")],
    )


## 3. Later: all six questions × 576 patches

Leave RUN_FULL_PILOT=False for now. After reviewing calibration, set RUN_FULL_PILOT=True in a separate code cell and run the next cell without rerunning setup. Estimate runtime from calibration seconds/question × 18 × 6 (approximate; prompts and target lengths differ). Keep the same frozen cases even if their correlations look uninteresting. Preserve all earlier results.


In [ ]:
if RUN_FULL_PILOT:
    if AUDIT_DIR is None or CALIBRATION_DIR is None:
        raise RuntimeError("Keep the passed audit and calibration folders available before the full pilot.")
    calibration = json.loads((Path(CALIBRATION_DIR) / "report.json").read_text())
    if (calibration["status"] != "passed"
            or calibration.get("case_ids") != ["textvqa:37533", "textvqa:35670"]
            or calibration.get("protocol", {}).get("tokens_measured_per_question") != 32):
        raise RuntimeError("Expected the passed two-question, 32-patch TextVQA calibration.")
    textvqa_pilot_dir = run_experiment(
        "rq1-textvqa-input-pilot-v1", "textvqa/manifest.json",
        ["--questions", "6", "--tokens", "576",
         "--target-audit-report", str(Path(AUDIT_DIR) / "report.json")],
    )


## Download before stopping

For this first run, download **both audit and calibration ZIPs and both logs** from Output. The six-case selection, original source/lock snapshots, full audit prerequisite, used images, token IDs, controls, native E diagnostics and scored patch effects are recorded. No weights are bundled. Keep all 100 original TextVQA screening predictions separately; this six-case subset is diagnostic and outcome-enriched, not a population accuracy sample.
